# Multi-Agent Patterns with LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/19-multi-agent-notebook/multi_agent_workbook.ipynb)

---

## What are multi-agent systems?

A **multi-agent system** coordinates multiple specialized LLM agents to tackle complex tasks that a single agent would handle poorly — either because the task is too broad, requires different tools, or benefits from specialization and cross-checking.

This workbook implements the **Supervisor/Worker** architecture — the most common production multi-agent pattern.

### Supervisor/Worker architecture

A central **supervisor** LLM decides which specialist worker should act next (or when the task is complete). Workers run independently, each with their own tools and system prompt.

```
START
  |
supervisor  <-----------+
  |                     |
  +-- next=researcher --+--> researcher_agent --> supervisor
  |                     |
  +-- next=writer    ---+--> writer_agent    --> supervisor
  |
  +-- next=FINISH --------> END
```

### What you will learn

- How `add_messages` reducer enables agent-to-agent message passing
- How `with_structured_output` gives the supervisor type-safe routing decisions
- How `create_react_agent` creates a fully functional worker agent in one call
- How `add_conditional_edges` wires the supervisor loop
- How to stream multi-agent execution step by step

## Cell 1 — Install dependencies

Only runs on Google Colab. Skip locally if you already have the packages.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q \
        langchain==0.3.27 \
        langchain-core==0.3.79 \
        langchain-openai==0.3.33 \
        langchain-community==0.3.31 \
        langgraph==0.6.7 \
        langgraph-prebuilt==0.6.4 \
        duckduckgo-search==8.1.1 \
        python-dotenv
    print('Dependencies installed.')
else:
    print('Local environment -- skipping install.')

## Cell 2 — API key

On Colab: enter your key when prompted. Locally: reads from `.env`.

In [ ]:
import os

if 'google.colab' in sys.modules:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY is not set!'
print('API key loaded.')

## Cell 3 — Shared state with `add_messages`

All agents share a single `AgentState`. The key design decision is the **message reducer**.

| Field | Type | Role |
|---|---|---|
| `messages` | `Annotated[list, add_messages]` | Shared conversation history; `add_messages` **appends** new messages rather than overwriting |
| `next` | `str` | Supervisor writes this to route to the next worker (`"researcher"`, `"writer"`, or `"FINISH"`) |

The `add_messages` reducer is what makes multi-agent message passing work: each worker appends its response to the shared list, so the supervisor always sees the full conversation when deciding what to do next.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # reducer: appends, never overwrites
    next: str                                # supervisor routing decision


print('State definition ready.')
print('  messages: add_messages reducer -- new messages are appended to the list')
print('  next:     supervisor writes this field to route to the correct worker')

## Cell 4 — Worker agents with `create_react_agent`

`create_react_agent` from `langgraph_prebuilt` creates a complete ReAct agent (observe-think-act loop) in a single call. You pass it an LLM and a list of tools — it handles the rest.

- **researcher**: equipped with DuckDuckGo web search
- **writer**: no tools — it synthesizes from the messages already in state

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model='gpt-5-nano')

search_tool = DuckDuckGoSearchResults(max_results=3)

researcher_agent = create_react_agent(
    llm,
    tools=[search_tool],
    prompt=SystemMessage(content=(
        'You are a research specialist. Search the web to find accurate, '
        'current information. Return a factual summary with key points.'
    )),
)

writer_agent = create_react_agent(
    llm,
    tools=[],
    prompt=SystemMessage(content=(
        'You are a writing specialist. Synthesize the research findings '
        'into a clear, well-structured, concise answer.'
    )),
)

print('Worker agents ready:')
print('  researcher_agent: gpt-5-nano + DuckDuckGo web search')
print('  writer_agent:     gpt-5-nano + no tools (synthesis only)')

## Cell 5 — The supervisor node

The supervisor is an LLM call that uses `with_structured_output` to force a typed routing decision. It sees the full message history and outputs a `SupervisorDecision` object with a `next` field.

Key patterns:

| Pattern | Purpose |
|---|---|
| `with_structured_output(SupervisorDecision)` | Forces the LLM to return valid JSON matching the Pydantic schema — no parsing, no regex |
| `Literal['researcher', 'writer', 'FINISH']` | Constrains routing to only valid next steps |
| Reading `state['messages']` | Supervisor has full visibility into what every previous agent said |

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel


class SupervisorDecision(BaseModel):
    next: Literal['researcher', 'writer', 'FINISH']


SUPERVISOR_PROMPT = ChatPromptTemplate.from_messages([
    ('system', """You are a supervisor coordinating two specialist agents:
- researcher: searches the web for current facts and information
- writer:     synthesizes research into a clear, structured answer

Rules:
1. Call researcher first to gather facts.
2. Call writer after researcher has returned findings.
3. Reply FINISH once the writer has produced a final answer.
4. Never call the same agent twice in a row."""),
    ('human', 'Conversation so far:\n{history}\n\nWho should act next?'),
])

supervisor_chain = SUPERVISOR_PROMPT | llm.with_structured_output(SupervisorDecision)

print('Supervisor ready.')
print('  Routes to: researcher | writer | FINISH')

## Cell 6 — Build the supervisor/worker graph

Graph structure:

1. **supervisor_node** — calls `supervisor_chain`, writes `next` to state
2. **researcher_node** / **writer_node** — invoke sub-agents, append their last message to state
3. **`add_conditional_edges`** on supervisor — reads `state['next']` to pick the next node
4. Both workers loop back to supervisor — the loop continues until supervisor returns `FINISH`

Notice that `researcher_node` and `writer_node` are thin wrappers: they call the prebuilt agent and extract just the final message to append to shared state.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph


def supervisor_node(state: AgentState) -> dict:
    history = '\n'.join(
        f'{m.type.upper()}: {m.content[:300]}'
        for m in state['messages']
    )
    decision = supervisor_chain.invoke({'history': history})
    return {'next': decision.next}


def researcher_node(state: AgentState) -> dict:
    result = researcher_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    # Tag the message so the supervisor knows which agent produced it
    tagged = AIMessage(content=f'[researcher] {last.content}', name='researcher')
    return {'messages': [tagged]}


def writer_node(state: AgentState) -> dict:
    result = writer_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    tagged = AIMessage(content=f'[writer] {last.content}', name='writer')
    return {'messages': [tagged]}


def route(state: AgentState) -> Literal['researcher', 'writer', '__end__']:
    nxt = state.get('next', 'FINISH')
    return '__end__' if nxt == 'FINISH' else nxt


graph = StateGraph(AgentState)
graph.add_node('supervisor', supervisor_node)
graph.add_node('researcher', researcher_node)
graph.add_node('writer', writer_node)

graph.add_edge(START, 'supervisor')
graph.add_conditional_edges('supervisor', route, {
    'researcher': 'researcher',
    'writer': 'writer',
    '__end__': END,
})
graph.add_edge('researcher', 'supervisor')
graph.add_edge('writer', 'supervisor')

app = graph.compile()
print('Supervisor/worker graph compiled.')
print('Nodes: supervisor -> {researcher, writer} -> supervisor -> ... -> END')

## Cell 7 — Visualise the graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'Graph render unavailable: {e}')
    print(app.get_graph().draw_mermaid())